# Lab 2: Regression

In this lab we implement linear, polynomial and regularized (ridge and lasso) regression from scratch with NumPy; robust regression on real data is a bonus to finish at home.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8")

## Plotting helpers

The two functions below only draw figures, so we do not spend time on plotting code. Run the two cells and move on.

**`plot_fits`** draws the data points and the prediction curve of one or several models.

In [ ]:
def plot_fits(x, y, predictors, ylim=None, title=None, ax=None):
    """Scatter the data (x, y) and draw one prediction curve per model.

    predictors: dict {label: f}, where f takes a 1D array of x values and returns the predicted y values
    ylim:       optional (y_min, y_max) limits of the vertical axis
    ax:         optional matplotlib axis, to draw inside a grid of subplots
    """
    # create a new figure if no axis is given
    if ax is None:
        ax = plt.figure(figsize=(8, 4)).gca()

    # data points
    ax.scatter(x, y, color="k", s=20, zorder=3, label="data")

    # one curve per model, evaluated on a fine grid of x values
    # (one line style per model, so that overlapping lines stay visible)
    x_grid = np.linspace(x.min(), x.max(), 500)
    line_styles = ["-", "--", ":", "-."]
    for k, (label, f) in enumerate(predictors.items()):
        ax.plot(x_grid, f(x_grid), lw=2, linestyle=line_styles[k % len(line_styles)], label=label)

    # axes, title and legend
    if ylim is not None:
        ax.set_ylim(*ylim)
    if title is not None:
        ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()

**`plot_weights`** colours each data point by the weight that robust regression gives it (bonus).

In [ ]:
def plot_weights(x, y, weights, title=None, ax=None):
    """Scatter the data (x, y), coloured by the weight of each point (0 = ignored, 1 = full weight).

    weights: (num_samples,) array, for example the final IRLS weights of a robust model
    ax:      optional matplotlib axis, to draw inside a grid of subplots
    """
    # create a new figure if no axis is given
    if ax is None:
        ax = plt.figure(figsize=(8, 4)).gca()

    # points coloured by their weight, with a colour bar to read the values
    points = ax.scatter(x, y, c=weights, cmap="viridis", vmin=0, vmax=1, s=40, edgecolor="k")
    plt.colorbar(points, ax=ax, label="weight")

    # axes and title
    if title is not None:
        ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

## 1. Linear regression

We look for the line $y = \beta_0 + \beta_1 x$ that best fits the data.
For each point, the **residual** is the difference between the observed value $y_i$ and the value of the line at $x_i$:
$$
e_i = y_i - (\beta_0 + \beta_1 x_i).
$$
It is measured along the $y$ axis, as on the "Least squares" slide (not perpendicular to the line): we want to predict $y$ from $x$.
Least squares chooses the line with the smallest **squared loss**:
$$
L(\beta_0, \beta_1) = \frac{1}{2} \sum_{i=1}^{n} e_i^2 = \frac{1}{2} \sum_{i=1}^{n} \left(y_i - \beta_0 - \beta_1 x_i\right)^2 .
$$
The $\frac{1}{2}$ does not change which line is the best.

We find this line in two ways: by trying many lines and keeping the best one (1.1), then exactly, with the normal equations (1.2).

We start with points scattered around a line with slope 2 and intercept −5.

In [ ]:
rng = np.random.RandomState(1)
x = rng.rand(30)
y = 2 * x - 5 + 0.1 * rng.randn(30)
plt.scatter(x, y)
plt.show()

### 1.1 Least squares: trying many lines

**Exercise 1a.** Complete `squared_loss`, which computes the loss of **one** line, point by point.

In [ ]:
def squared_loss(beta0, beta1, x, y):
    """Squared loss of the line y = beta0 + beta1 * x on the data (x, y):
    L = 1/2 * sum_i (y_i - (beta0 + beta1 * x_i))^2

    Parameters:
    -----------
    beta0, beta1: float
        Intercept and slope of the line
    x, y: (num_samples,) np.array
        Data points

    Returns:
    --------
    loss: float
    """
    # TODO
    # Step 1: start the sum of squared residuals at 0
    # Step 2: for every point i:
    #     a) value of the line at x_i: prediction = beta0 + beta1 * x_i
    #     b) residual: y_i - prediction
    #     c) add the squared residual to the sum
    # Step 3: return half of the sum

Test the loss on two lines: the correct one (intercept −5, slope 2) and a wrong one (intercept −4, slope 1). The correct line should have a much smaller loss.

In [ ]:
print("Loss of the correct line (intercept -5, slope 2):", squared_loss(-5, 2, x, y))
print("Loss of a wrong line     (intercept -4, slope 1):", squared_loss(-4, 1, x, y))

The loss tells us which of two lines is better. Instead of testing lines by hand, we let the computer try many lines and keep the best one.

**Exercise 1b.** Complete `least_squares_search`: it calls `squared_loss` for **every** combination of candidate intercepts and slopes, and keeps the line with the smallest loss.

In [ ]:
def least_squares_search(x, y, intercepts, slopes):
    """Try every line y = beta0 + beta1 * x with beta0 in intercepts and beta1 in slopes,
    and return the one with the smallest squared loss.

    Parameters:
    -----------
    x, y: (num_samples,) np.array
        Data points
    intercepts, slopes: 1D np.array
        Candidate values for beta0 and beta1

    Returns:
    --------
    best_intercept, best_slope: float
    """
    # TODO
    # Step 1: start with best_loss = infinity (np.inf), so that the first line tried becomes the best one
    # Step 2: for every intercept beta0 in intercepts and every slope beta1 in slopes (two nested loops):
    #     a) compute the loss of this line with squared_loss
    #     b) if this loss is smaller than best_loss, remember this loss, this intercept and this slope
    # Step 3: return best_intercept, best_slope

Search the best line and plot it:

In [ ]:
intercepts = np.linspace(-6, -4, 201)  # candidate intercepts, step 0.01
slopes = np.linspace(1, 3, 201)        # candidate slopes, step 0.01
search_intercept, search_slope = least_squares_search(x, y, intercepts, slopes)

print("Intercept:", round(search_intercept, 3))
print("Slope:    ", round(search_slope, 3))
plot_fits(x, y, {"least squares (search)": lambda t: search_intercept + search_slope * t})
plt.show()

### 1.2 Least squares: the normal equations

Trying many lines is slow, and only as precise as the candidate values. The normal equations give the best line directly.

**Predictions as one matrix product.** We add a column of ones to the left of the data and call this matrix $X$ (in the code: `X_aug = self.add_intercept(X)`).
Row $i$ of $X$ is $(1, x_i)$, so with $\beta = (\beta_0, \beta_1)$ the predictions of all the points are
$$
\hat y = X \beta, \qquad \text{i.e.} \qquad \hat y_i = \beta_0 + \beta_1 x_i .
$$

**The best line.** The squared loss of 1.1 becomes $L(\beta) = \frac{1}{2}\|y - X\beta\|^2$.
At its minimum the derivative is zero, which gives the **normal equations** and their solution:
$$
X^T X\, \beta = X^T y
\qquad \Longrightarrow \qquad
\beta = \left(X^T X\right)^{-1} X^T y .
$$

**Exercise 2.** Complete the `LinearRegression` class.

`add_intercept` and `predict` are reused by every model of this lab: the next classes inherit from `LinearRegression` and only need a new `fit`.

In [ ]:
class LinearRegression:
    """Ordinary least squares: minimises 1/2 ||y - X beta||^2."""

    def __init__(self):
        self.coef_ = None  # will hold [beta_0, beta_1, ..., beta_d]

    def add_intercept(self, X):
        """Return X with a first column of ones.

        Parameters:
        -----------
        X: (num_samples, num_features) np.array
            Design matrix

        Returns:
        --------
        X_aug: (num_samples, num_features + 1) np.array
        """
        # TODO
        # Step 1: get the number of rows (samples) of X (X.shape)
        # Step 2: create a column of ones of shape (num_samples, 1) (np.ones)
        # Step 3: put this column to the left of X (np.hstack) -> X_aug, shape (num_samples, num_features + 1)
        # Step 4: return X_aug

    def fit(self, X, y):
        """Fit the data (X, y) and store the coefficients in self.coef_.

        Parameters:
        -----------
        X: (num_samples, num_features) np.array
            Design matrix
        y: (num_samples,) np.array
            Output vector
        """
        # TODO
        # Step 1: add the column of ones for the intercept: X_aug = self.add_intercept(X)
        # Step 2: left-hand side of the normal equations: XtX = X_aug^T X_aug, shape (num_features + 1, num_features + 1)
        # Step 3: right-hand side of the normal equations: Xty = X_aug^T y, shape (num_features + 1,)
        # Step 4: solve XtX beta = Xty, i.e. beta = (X_aug^T X_aug)^{-1} X_aug^T y
        #         tip: np.linalg.solve(XtX, Xty) gives the same result as np.linalg.inv(XtX) @ Xty,
        #         but it is faster and more accurate
        # Step 5: store beta in self.coef_ (beta[0] is the intercept, beta[1:] are the slopes)
        return self

    def predict(self, X):
        """Make predictions for data X.

        Parameters:
        -----------
        X: (num_samples, num_features) np.array
            Design matrix

        Returns:
        --------
        y_pred: (num_samples,) np.array
            Predictions
        """
        # TODO
        # Step 1: add the column of ones, as in fit: X_aug = self.add_intercept(X)
        # Step 2: compute y_pred = X_aug @ self.coef_ (one prediction per sample)
        # Step 3: return y_pred

Fit the model and plot both lines (`x[:, None]` turns the 1D array `x` into a matrix with one column):

In [ ]:
model = LinearRegression()
model.fit(x[:, None], y)

print("                    intercept   slope")
print(f"search              {search_intercept:9.3f} {search_slope:7.3f}")
print(f"normal equations    {model.coef_[0]:9.3f} {model.coef_[1]:7.3f}")

plot_fits(x, y, {"least squares (search)": lambda t: search_intercept + search_slope * t,
                 "least squares (normal equations)": lambda t: model.predict(t[:, None])})
plt.show()

Both methods find almost the same line, close to the true slope 2 and intercept −5.

## 2. Polynomial regression

A linear model can fit non-linear relationships if we first transform the input with *basis functions*.
With $x_k = x^k$ the model becomes a polynomial:
$$
y = \beta_0 + \beta_1 x + \beta_2 x^2 + \dots + \beta_p x^p .
$$
This is *still a linear model*: it is linear in the coefficients $\beta$, not in $x$.
We only need to build the matrix of powers of $x$ and give it to `LinearRegression`.

**Exercise 3.** Complete `polynomial_features`.

In [ ]:
def polynomial_features(x, degree):
    """Return the matrix of powers [x, x^2, ..., x^degree].

    Parameters:
    -----------
    x: (num_samples,) np.array
    degree: int

    Returns:
    --------
    X_poly: (num_samples, degree) np.array
        Column k-1 contains x**k. No column of ones: the model already adds the intercept.
    """
    # TODO
    # Step 1: create an empty list that will hold the columns
    # Step 2: for every power k from 1 to degree, append the column x**k to the list
    # Step 3: stack the columns side by side (np.column_stack) -> X_poly, shape (num_samples, degree)
    # Step 4: return X_poly

Check `polynomial_features` on three values, $x = 2, 3, 4$, with degree 3. Each row should be $[x, x^2, x^3]$, that is $[2, 4, 8]$, $[3, 9, 27]$ and $[4, 16, 64]$:

In [ ]:
print(polynomial_features(np.array([2, 3, 4]), 3))

Now fit a noisy sine wave with polynomials of increasing degree:

In [ ]:
rng = np.random.RandomState(1)
x = rng.rand(20)
y = 2 * np.sin(1.8 * np.pi * x) + 0.1 * rng.randn(20)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, degree in zip(axes, [3, 7, 15]):
    model = LinearRegression()
    model.fit(polynomial_features(x, degree), y)
    plot_fits(x, y, {f"degree {degree}": lambda t: model.predict(polynomial_features(t, degree))},
              ylim=(-3, 3), ax=ax)

Degree 7 fits the sine well. Degree 15 passes close to every point but oscillates between them: with 20 points, the model has too much flexibility and **overfits**.

**Question.** How can we avoid overfitting?

**Answer:**

## 3. Regularization

Look at the size of the degree-15 coefficients:

In [ ]:
X_poly = polynomial_features(x, 15)
ols = LinearRegression()
ols.fit(X_poly, y)
print("Largest |coefficient|:", np.abs(ols.coef_).max())

Huge coefficients that cancel each other produce the oscillations. Regularization adds a penalty that keeps $\beta$ small:
$$
L_{new}(\beta) = \frac{1}{2}\|y - X\,\beta\|^2 + \lambda\, R(\beta).
$$
$\lambda = 0$ gives least squares back, and $\lambda \to \infty$ crushes the coefficients to 0.
The intercept $\beta_0$ is **not** penalized: it only sets the level of the curve.

### 3.1 Ridge regression ($L_2$ penalty)

With $R(\beta) = \frac{1}{2}\|\beta\|^2$ the loss stays quadratic and the solution is explicit:
$$
\beta = \left(X^T X + \lambda I_0\right)^{-1} X^T\, y,
$$
where $I_0$ is the identity matrix with a 0 in the top-left corner, so that $\beta_0$ is not penalized.

**Exercise 4.** Complete `RidgeRegression`. It inherits `add_intercept` and `predict` from `LinearRegression`, so only `fit` changes.

In [ ]:
class RidgeRegression(LinearRegression):
    """Least squares with an L2 penalty: minimises 1/2 ||y - X beta||^2 + lam/2 ||beta||^2.
    The intercept is not penalized."""

    def __init__(self, lam):
        super().__init__()
        self.lam = lam  # lambda, the strength of the penalty

    def fit(self, X, y):
        """Fit the data (X, y) and store the coefficients in self.coef_.

        Parameters:
        -----------
        X: (num_samples, num_features) np.array
            Design matrix
        y: (num_samples,) np.array
            Output vector
        """
        # TODO
        # Step 1: add the column of ones for the intercept: X_aug = self.add_intercept(X)
        # Step 2: XtX = X_aug^T X_aug and Xty = X_aug^T y, as in LinearRegression
        # Step 3: penalty matrix lam * I, of size (num_features + 1) x (num_features + 1) (np.eye)
        # Step 4: set its top-left entry penalty[0, 0] to 0, so that the intercept beta_0 is not penalized
        # Step 5: solve (XtX + penalty) beta = Xty (np.linalg.solve)
        # Step 6: store beta in self.coef_
        return self

Compare least squares and ridge on the degree-15 polynomial:

In [ ]:
ridge = RidgeRegression(lam=1e-3)
ridge.fit(X_poly, y)

plot_fits(x, y, {"least squares": lambda t: ols.predict(polynomial_features(t, 15)),
                 "ridge, λ = 1e-3": lambda t: ridge.predict(polynomial_features(t, 15))},
          ylim=(-3, 3))
print("Largest |coefficient|, least squares:", np.abs(ols.coef_).max())
print("Largest |coefficient|, ridge:        ", np.abs(ridge.coef_).max())

$\lambda$ is a knob that controls the complexity of the model:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, lam in zip(axes, [0, 1e-3, 1]):
    ridge = RidgeRegression(lam)
    ridge.fit(X_poly, y)
    plot_fits(x, y, {f"ridge, λ = {lam:g}": lambda t: ridge.predict(polynomial_features(t, 15))},
              ylim=(-3, 3), ax=ax)

$\lambda = 0$ is least squares and overfits; a large $\lambda$ flattens the curve and underfits.

**Question.** How can we choose $\lambda$?

**Answer:**

### 3.2 Lasso regression ($L_1$ penalty)

With $R(\beta) = \|\beta\|_1 = \sum_j |\beta_j|$, the penalty favours **sparse** models: it sets coefficients exactly to zero.
The penalty is not differentiable at 0, so there is no closed-form solution. We use the **proximal gradient** algorithm (ISTA) from the lecture, which repeats two steps:

1. **gradient step** on the smooth part: $z = \beta^k - \theta\, X^T (X\,\beta^k - y)$
2. **proximal step** on the penalty, which for $L_1$ is a **soft thresholding**: $\beta^{k+1} = \operatorname{sign}(z)\, \max(|z| - \lambda\theta,\ 0)$

The step size $\theta = 1 / \|X^T X\|$ (largest eigenvalue of $X^T X$) guarantees convergence.
As for ridge, the intercept is not penalized: it gets the gradient step but not the thresholding.

**Exercise 5.** Complete `LassoRegression`.

In [ ]:
class LassoRegression(LinearRegression):
    """Least squares with an L1 penalty: minimises 1/2 ||y - X beta||^2 + lam ||beta||_1,
    solved by proximal gradient descent (ISTA). The intercept is not penalized."""

    def __init__(self, lam, n_iter=100000):
        super().__init__()
        self.lam = lam        # lambda, the strength of the penalty
        self.n_iter = n_iter  # number of ISTA iterations

    def soft_threshold(self, z, t):
        """Soft thresholding sign(z) * max(|z| - t, 0), applied to every entry of the array z."""
        # TODO
        # Step 1: shrink the absolute values by t and set to 0 those that become negative: np.maximum(np.abs(z) - t, 0)
        # Step 2: put the original signs back (multiply by np.sign(z)) and return the result

    def fit(self, X, y):
        """Fit the data (X, y) and store the coefficients in self.coef_.

        Parameters:
        -----------
        X: (num_samples, num_features) np.array
            Design matrix
        y: (num_samples,) np.array
            Output vector
        """
        # TODO
        # Step 1: add the column of ones for the intercept: X_aug = self.add_intercept(X)
        # Step 2: XtX = X_aug^T X_aug and Xty = X_aug^T y (computed once, used at every iteration)
        # Step 3: step size theta = 1 / ||XtX||, the largest eigenvalue of XtX (np.linalg.norm(XtX, 2))
        # Step 4: start from beta = 0, one entry per column of X_aug (np.zeros)
        # Step 5: repeat self.n_iter times:
        #     a) gradient of the smooth part 1/2 ||y - X_aug beta||^2: gradient = XtX beta - Xty
        #     b) gradient step: z = beta - theta * gradient
        #     c) proximal step: beta = z.copy(), then soft threshold every entry except the intercept:
        #        beta[1:] = self.soft_threshold(z[1:], self.lam * theta)
        # Step 6: store beta in self.coef_
        return self

Compare least squares, ridge and lasso on the degree-15 polynomial of section 2.
To see which model is better, we measure the error on new points from the same curve (test points), which are not used for fitting:

In [ ]:
# new points from the same curve, in the same range as the training points
rng = np.random.RandomState(2)
x_test = rng.uniform(x.min(), x.max(), 200)
y_test = 2 * np.sin(1.8 * np.pi * x_test) + 0.1 * rng.randn(200)

models = {"least squares": LinearRegression(),
          "ridge, λ = 1e-3": RidgeRegression(lam=1e-3),
          "lasso, λ = 1e-3": LassoRegression(lam=1e-3)}
for name, m in models.items():
    m.fit(X_poly, y)
    # root mean squared error on the test points
    test_error = np.sqrt(np.mean((m.predict(polynomial_features(x_test, 15)) - y_test) ** 2))
    n_zeros = np.sum(m.coef_[1:] == 0)
    print(f"{name:16s} test error = {test_error:.3f}   coefficients equal to 0: {n_zeros} / 15")

plot_fits(x, y, {name: (lambda t, m=m: m.predict(polynomial_features(t, 15))) for name, m in models.items()},
          ylim=(-3, 3))
plt.show()

Both penalties fix the overfitting of least squares. With the same $\lambda$, lasso is slightly better here, and it also sets some coefficients exactly to 0: it keeps only part of the powers of $x$.

## Bonus (at home): robust regression

### Outliers

Least squares squares every residual: a point twice as far away counts four times as much, so a few outliers can pull the whole fit.
Here the noise is heavy-tailed (Cauchy), which produces a few points far from the true line $y = 2x - 5$:

In [ ]:
rng = np.random.RandomState(76)
x = rng.rand(30)
y = 2 * x - 5 + 0.1 * rng.standard_cauchy(30)
plt.scatter(x, y)
plt.show()

**M-estimation** replaces the square by a loss $\rho$ that grows more slowly for large residuals:
$$
\min_\beta \sum_i \rho\left(\frac{y_i - \hat y_i}{\hat\sigma}\right), \qquad \hat y = X\,\beta .
$$
Residuals are divided by a **robust scale** $\hat\sigma = 1.4826 \times \mathrm{MAD}$, with $\mathrm{MAD} = \mathrm{median}_i\, |e_i - \mathrm{median}(e)|$.
This way the threshold $\delta$ is measured in units of the residual spread, and the outliers cannot inflate the scale.

Setting the gradient to zero gives a least-squares problem with one weight per point, $\omega(r) = \dot\rho(r) / r$.
For the **Huber** loss ($\delta = 1.345$):
$$
\omega(r) = \begin{cases} 1 & \text{if } |r| \le \delta \\ \delta / |r| & \text{if } |r| > \delta \end{cases}
$$
Ordinary points keep weight 1 and far points get smaller weights.
The weights depend on the residuals, which depend on $\beta$, so we alternate (**IRLS**, iteratively reweighted least squares):

1. start from the least-squares fit $\beta^0$
2. residuals $e = y - X\,\beta^k$ and scale $\hat\sigma = 1.4826 \times \mathrm{MAD}(e)$
3. weights $w_i = \omega(e_i / \hat\sigma)$
4. weighted least squares $\beta^{k+1} = \left(X^T W X\right)^{-1} X^T W y$, with $W = \mathrm{diag}(w)$

**Bonus exercise.** Complete `RobustRegression` with the Huber weights.

*Optional:* add the **Tukey** bisquare weights ($\delta = 4.685$): $\omega(r) = \left(1 - (r/\delta)^2\right)^2$ if $|r| \le \delta$, and 0 otherwise.

In [ ]:
class RobustRegression(LinearRegression):
    """M-estimation solved by iteratively reweighted least squares (IRLS)."""

    def __init__(self, potential="huber", delta=1.345, n_iter=20):
        super().__init__()
        self.potential = potential  # "huber" or "tukey"
        self.delta = delta          # threshold δ, in units of the robust scale
        self.n_iter = n_iter        # number of IRLS iterations
        self.weights_ = None        # final weight of each point, filled by fit

    def mad(self, e):
        """Median absolute deviation of the residuals: median(|e - median(e)|)."""
        # TODO
        # Step 1: median of the residuals (np.median)
        # Step 2: absolute deviations from this median, |e - median(e)| (np.abs)
        # Step 3: return the median of these absolute deviations

    def weight_function(self, r):
        """Weight of every scaled residual r = e / sigma (r is an array)."""
        if self.potential == "huber":
            # TODO
            # Step 1: absolute values |r| (np.abs)
            # Step 2: weight 1 where |r| <= delta, and delta / |r| elsewhere (np.where), and return it
            pass
        elif self.potential == "tukey":
            # Optional TODO
            # Step 1: absolute values |r| (np.abs)
            # Step 2: weight (1 - (r / delta)**2)**2 where |r| <= delta, and 0 elsewhere (np.where), and return it
            pass

    def fit(self, X, y):
        """Fit the data (X, y) and store the coefficients in self.coef_.

        Parameters:
        -----------
        X: (num_samples, num_features) np.array
            Design matrix
        y: (num_samples,) np.array
            Output vector
        """
        # TODO
        # Step 1: add the column of ones for the intercept: X_aug = self.add_intercept(X)
        # Step 2: initialise beta with ordinary least squares (no weights), as in LinearRegression
        # Step 3: repeat self.n_iter times:
        #     a) residuals of the current fit: e = y - X_aug beta
        #     b) robust scale of the residuals: sigma = 1.4826 * self.mad(e)
        #     c) scaled residuals r = e / sigma, and their weights w = self.weight_function(r)
        #     d) weighted normal equations, with W = diag(w): XtWX = X_aug^T W X_aug and XtWy = X_aug^T W y
        #        tip: X_aug.T * w gives X_aug^T W without building the n x n matrix W
        #     e) solve XtWX beta = XtWy (np.linalg.solve)
        # Step 4: store beta in self.coef_ and w in self.weights_
        return self

Compare least squares, Huber and Tukey on the data with outliers (remove the Tukey line if you skipped the optional part):

In [ ]:
models = {"least squares": LinearRegression(),
          "Huber": RobustRegression("huber", delta=1.345),
          "Tukey": RobustRegression("tukey", delta=4.685)}
for name, m in models.items():
    m.fit(x[:, None], y)
    print(f"{name:14s} slope = {m.coef_[1]:.2f}   intercept = {m.coef_[0]:.2f}   (true: 2, -5)")

plot_fits(x, y, {name: (lambda t, m=m: m.predict(t[:, None])) for name, m in models.items()})

The final IRLS weights show which points the robust fits trust:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_weights(x, y, models["Huber"].weights_, title="Huber weights", ax=axes[0])
plot_weights(x, y, models["Tukey"].weights_, title="Tukey weights", ax=axes[1])

Least squares is pulled away by the outliers. Huber and Tukey give them small weights and recover the true line, without any hand-made outlier detection.
Tukey's weights drop to exactly 0, so the far points are ignored completely; Huber's weights only decrease as $\delta / |r|$.

### Predicting bicycle traffic

We now apply least squares and Huber regression to real data: the daily number of bicycle trips across Seattle's Fremont Bridge, predicted from the day of the week, holidays, daylight hours, precipitation (`PRCP`), temperature and a yearly trend (`annual`).
Time-series methods would be more appropriate for this data; here, as a first simple approach, we only use the regression models of this lab.

Note: the seven day columns `Mon`, ..., `Sun` always sum to 1, which is exactly the column of ones of the intercept.
Keeping all seven would make $X^T X$ non-invertible (the same information appears twice), so we drop `Sun`.

In [ ]:
daily = pd.read_csv("data.csv", index_col="Date", parse_dates=True)
daily = daily.dropna()

features = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "holiday",
            "daylight_hrs", "PRCP", "dry day", "Temp (C)", "annual"]
X = daily[features].values
y = daily["Total"].values

# fit both models and store their predictions
ols = LinearRegression()
ols.fit(X, y)
huber = RobustRegression("huber", delta=1.345)
huber.fit(X, y)
daily["least squares"] = ols.predict(X)
daily["Huber"] = huber.predict(X)

# root mean squared error, in trips per day
for name in ["least squares", "Huber"]:
    rmse = np.sqrt(np.mean((daily[name] - daily["Total"]) ** 2))
    print(f"{name:14s} RMSE = {rmse:.0f} trips per day")

# effect of each feature on the number of trips (Sunday is the reference day)
print(pd.DataFrame({"least squares": ols.coef_[1:], "Huber": huber.coef_[1:]}, index=features).round(1))

Compare the true and predicted traffic:

In [ ]:
daily[["Total", "least squares", "Huber"]].plot(alpha=0.5, figsize=(16, 4))
plt.show()

Both models give almost the same predictions, and both miss some key features, especially in summer.
Either our features are not complete (people decide whether to ride to work based on more than these), or there are non-linear relationships we did not take into account (for example, people may ride less at both high and low temperatures).
In that case the *model* is wrong, and robust regression cannot fix it: it only protects against a few wrong *points*.

Which days does Huber trust the least?

In [ ]:
daily["weight"] = huber.weights_
lowest = daily.nsmallest(8, "weight")[["Total", "Huber", "weight"]]
lowest.insert(0, "day", lowest.index.day_name())
print(lowest.round(2))

**Question.** What do these days have in common? Why does the model get them wrong?

**Answer:**

Robust regression helps when some points are wrong. We simulate a broken counter: 30 random days are recorded as 0 trips.
We fit both models on the broken data and measure their error on the **true** counts.

In [ ]:
rng = np.random.RandomState(0)
broken_days = rng.choice(len(y), size=30, replace=False)
y_broken = y.copy()
y_broken[broken_days] = 0

ols_broken = LinearRegression()
ols_broken.fit(X, y_broken)
huber_broken = RobustRegression("huber", delta=1.345)
huber_broken.fit(X, y_broken)

for name, clean, broken in [("least squares", ols, ols_broken), ("Huber", huber, huber_broken)]:
    rmse_clean = np.sqrt(np.mean((clean.predict(X) - y) ** 2))
    rmse_broken = np.sqrt(np.mean((broken.predict(X) - y) ** 2))
    print(f"{name:14s} RMSE on the true counts: {rmse_clean:.0f} (clean data) -> {rmse_broken:.0f} (broken counter)")
print("Mean Huber weight of the broken days:", huber_broken.weights_[broken_days].mean().round(2))

Least squares is pulled towards the zeros and predicts the true traffic worse. Huber gives the broken days small weights, so its model barely changes.